# E2-A: Feature Dictionary

**Spec sections**: 2.4 (Feature Dictionary), 2.5 (Assigning Roles to Columns)

Classify every column in the source table by the rules in spec 2.5, build
`config/feature_dict.csv` with the fields in spec 2.4, and stop if any column
is UNASSIGNED.

**No label-based statistics are computed in this part.**

In [0]:
import sys, os

# Project root (one level up from src/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import common

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
print(f"Project: {cfg['project']}  |  Config hash: {common.config_hash(cfg)[:12]}")
print(f"Source table: {cfg['data']['raw_snapshot_table']}")

In [0]:
# Read the raw snapshot table (Appendix F: read Delta with Spark, then .toPandas())
table = cfg["data"]["raw_snapshot_table"]
sdf = spark.table(table)

# Capture Spark schema for dtype mapping
schema_fields = {f.name: str(f.dataType) for f in sdf.schema.fields}
print(f"Source table: {table}")
print(f"Rows: {sdf.count():,}  |  Columns: {len(sdf.columns)}")

# Convert to pandas for further analysis
raw = sdf.toPandas()
print(f"Pandas shape: {raw.shape}")

In [0]:
# Use common.assign_column_roles which implements the spec 2.5 priority rules:
# KEY > PERFORMANCE > FLAG > WATCH (if not in override) > FEATURE_OVERRIDE > PREFIX > UNASSIGNED
roles_df = common.assign_column_roles(list(raw.columns), cfg)

print(f"\n{'='*60}")
print("COLUMN ROLE COUNTS (spec 2.5)")
print(f"{'='*60}")
role_counts = roles_df["role"].value_counts()
for role, cnt in role_counts.items():
    print(f"  {role:20s}  {cnt}")
print(f"  {'TOTAL':20s}  {len(roles_df)}")

In [0]:
# Print every column with role KEY, PERFORMANCE, FLAG, WATCH, UNASSIGNED
for role_name in ["KEY", "PERFORMANCE", "FLAG", "WATCH", "UNASSIGNED"]:
    subset = roles_df[roles_df["role"] == role_name]
    if len(subset) == 0:
        print(f"\n{role_name}: (none)")
        continue
    print(f"\n{role_name} ({len(subset)} columns):")
    for _, row in subset.iterrows():
        print(f"  {row['feature']:40s}  rule: {row['rule']}")

In [0]:
# Spec 2.5: any UNASSIGNED column stops the pipeline
unassigned = roles_df[roles_df["role"] == "UNASSIGNED"]
if len(unassigned) > 0:
    print("\n" + "!"*60)
    print(f"PIPELINE STOPPED: {len(unassigned)} UNASSIGNED column(s).")
    print("Add them to column_roles.key, .performance, .watch,")
    print(".feature_override, or use a recognised prefix.")
    print("!"*60)
    for _, row in unassigned.iterrows():
        print(f"  - {row['feature']}")
    raise SystemExit(f"{len(unassigned)} UNASSIGNED columns")
else:
    print("\nNo UNASSIGNED columns — pipeline may proceed.")

In [0]:
# Map Spark types to spec 2.4 dtype: numerical / categorical
_NUMERICAL_TYPES = {
    "IntegerType", "LongType", "ShortType", "ByteType",
    "FloatType", "DoubleType", "DecimalType",
}

def spark_type_to_dtype(spark_type_str: str) -> str:
    """Map a Spark type string to 'numerical' or 'categorical'."""
    # Handle DecimalType(precision, scale) and similar parameterised types
    base = spark_type_str.split("(")[0].strip()
    if base in _NUMERICAL_TYPES:
        return "numerical"
    return "categorical"

roles_df["dtype"] = roles_df["feature"].map(
    lambda c: spark_type_to_dtype(schema_fields.get(c, "StringType"))
)

# Post-hoc override: StringType FEATURE columns where ALL non-null values
# are numeric should be dtype="numerical" (e.g. aecb_AECB_Score stored as
# string in source).  ID-like columns (name contains 'id', 'name', 'iban')
# are excluded — they happen to be numeric but are identifiers, not features.
_ID_TOKENS = {"id", "name", "iban", "ref", "no"}
overrides = []
for idx, row in roles_df.iterrows():
    if row["role"] != "FEATURE" or row["dtype"] != "categorical":
        continue
    spark_t = schema_fields.get(row["feature"], "")
    if "String" not in spark_t:
        continue
    # Skip ID-like columns
    low = row["feature"].lower()
    if any(tok in low.split("_") for tok in _ID_TOKENS):
        continue
    s = raw[row["feature"]].dropna()
    if len(s) == 0:
        continue
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().sum() == len(s):  # 100% convertible
        roles_df.at[idx, "dtype"] = "numerical"
        overrides.append(row["feature"])

if overrides:
    print(f"dtype overrides (StringType -> numerical): {overrides}")

print("\ndtype distribution:")
print(roles_df["dtype"].value_counts().to_string())

In [0]:
# Derive source field (spec 2.4): AECB / FTS / CASA / APP
def derive_source(col_name: str) -> str:
    low = col_name.lower()
    if low.startswith("aecb_"):
        return "AECB"
    if low.startswith("fts_"):
        return "FTS"
    if low.startswith("casa_"):
        return "CASA"
    if low.startswith("app_"):
        return "APP"
    return ""

roles_df["source"] = roles_df["feature"].map(derive_source)
print("source distribution (FEATURE columns only):")
print(roles_df.loc[roles_df["role"] == "FEATURE", "source"].value_counts().to_string())

In [0]:
# Spec 2.4 / 2.5: propose special_codes from values such as -999, -998, -1
# that occur in >= 0.5% of records.  Each proposal MUST be confirmed or removed.
SENTINEL_CANDIDATES = [-999, -998, -997, -1, 9999, 99999]
MIN_SHARE = 0.005  # 0.5%

feature_cols = roles_df.loc[roles_df["role"] == "FEATURE", "feature"].tolist()
numerical_features = [
    c for c in feature_cols
    if roles_df.loc[roles_df["feature"] == c, "dtype"].iloc[0] == "numerical"
]

n_rows = len(raw)
special_codes_map = {}  # col -> list of detected sentinels

for col in numerical_features:
    vals = pd.to_numeric(raw[col], errors="coerce")
    detected = []
    for sv in SENTINEL_CANDIDATES:
        share = (vals == sv).sum() / n_rows
        if share >= MIN_SHARE:
            detected.append(sv)
    if detected:
        special_codes_map[col] = detected

print(f"Columns with proposed special codes: {len(special_codes_map)}")
for col, codes in sorted(special_codes_map.items()):
    pcts = [f"{sv} ({(pd.to_numeric(raw[col], errors='coerce') == sv).sum() / n_rows:.1%})" for sv in codes]
    print(f"  {col:45s}  {', '.join(pcts)}")

In [0]:
# Build the full feature dictionary per spec 2.4
fdict = roles_df[["feature", "role", "rule"]].copy()
fdict["source"] = roles_df["source"]
fdict["dtype"] = roles_df["dtype"]
fdict["description"] = ""  # to be filled by model owner
fdict["special_codes"] = fdict["feature"].map(
    lambda c: str(special_codes_map.get(c, ""))
)
fdict["special_meaning"] = ""  # to be filled after confirmation
fdict["available_at_decision"] = "Y"  # default; exceptions set individually

# compliance_ok: Y for FEATURE, N for WATCH (sensitive), empty otherwise
def compliance_flag(row):
    if row["role"] == "FEATURE":
        return "Y"
    if row["role"] == "WATCH":
        return "N"
    return ""

fdict["compliance_ok"] = fdict.apply(compliance_flag, axis=1)
fdict["expected_direction"] = "?"  # binning chooses direction
fdict["definition_change_date"] = ""  # to be filled if applicable

# Drop the internal 'rule' column used for auditing (keep for traceability)
# Reorder to match spec 2.4
final_cols = [
    "feature", "role", "source", "dtype", "description",
    "special_codes", "special_meaning", "available_at_decision",
    "compliance_ok", "expected_direction", "definition_change_date", "rule",
]
fdict = fdict[final_cols]

# Write to config/feature_dict.csv
out_path = os.path.join(PROJECT_ROOT, "config", "feature_dict.csv")
fdict.to_csv(out_path, index=False)
print(f"Written: {out_path}")
print(f"Rows: {len(fdict)}  |  FEATURE columns: {(fdict['role'] == 'FEATURE').sum()}")
print(f"\nFirst 10 rows:")
fdict.head(10)

In [0]:
# Final summary
print("="*60)
print("E2-A FEATURE DICTIONARY COMPLETE")
print("="*60)
print(f"\nTable read: {cfg['data']['raw_snapshot_table']}")
print(f"Total columns: {len(fdict)}")
print(f"\nRole counts:")
for role, cnt in fdict["role"].value_counts().sort_index().items():
    print(f"  {role:20s}  {cnt}")
print(f"\nSpecial-code proposals: {len(special_codes_map)} columns")
print(f"Output: config/feature_dict.csv")
print(f"\nNote: special_codes are PROPOSALS — each must be confirmed")
print(f"      or removed by the model owner (spec 2.4).")